# `collections` — Specialized Container Datatypes

The `collections` module provides specialized container datatypes that extend or replace Python's general-purpose built-ins (`dict`, `list`, `set`, `tuple`).

This notebook covers:
1. **`deque`** (double-ended queue + bounded ring buffer with `maxlen`)
2. **`Counter`** (hashable frequency counting & multiset algebra)
3. **`OrderedDict`** (what still distinguishes it from `dict` in Python 3.7+ — e.g. LRU caches)
4. **`defaultdict`** (default factory patterns & read-mutation gotcha)
5. **`namedtuple`** (lightweight immutable record types)
6. **`ChainMap`** (layered dictionary lookups without copying)
7. **Time Complexity Summary**

In [1]:
from collections import ChainMap, Counter, OrderedDict, defaultdict, deque, namedtuple

---
## 1. `deque` (Double-Ended Queue)

Pronounced *"deck"*, a `deque` is implemented in CPython as a **doubly-linked list of fixed-size memory blocks** (64 elements per block).

| Operation | `deque` | `list` |
|---|---|---|
| `append(x)` / `pop()` | $O(1)$ | $O(1)$ amortized |
| `appendleft(x)` / `popleft()` | $O(1)$ | $O(n)$ (shifts all elements) |
| Access/modify ends (`d[0]`, `d[-1]`) | $O(1)$ | $O(1)$ |
| Random access/modify middle (`d[i]`) | **$O(n)$** (traverses blocks) | **$O(1)$** |
| `extend(it)` / `extendleft(it)` | $O(k)$ | $O(k)$ / $O(n + k)$ |
| `remove(x)` / `index(x)` / `count(x)` | $O(n)$ | $O(n)$ |
| `reverse()` | **$O(n)$** (swaps elements in-place) | $O(n)$ |
| `rotate(k)` | $O(\min(|k|, n - |k|))$ | N/A |

> **Key takeaway:** Use `deque` for queues, BFS, sliding windows, and fixed-size rolling histories. Use `list` when you need fast random indexing (`arr[i]`) or in-place `.sort()` (`deque` has no `.sort()` method).

```
                           ┌─────┬─────┬─────┬─────┬─────┐
0. Initial deque:          │ 10  │ 20  │ 30  │ 40  │ 50  │
                           └─────┴─────┴─────┴─────┴─────┘
                           ┌─────┬─────┬─────┬─────┬─────┬─────┐
1. append(60):             │ 10  │ 20  │ 30  │ 40  │ 50  │ 60  │
                           └─────┴─────┴─────┴─────┴─────┴─────┘
                     ┌─────┬─────┬─────┬─────┬─────┬─────┬─────┐
2. appendleft(5):    │  5  │ 10  │ 20  │ 30  │ 40  │ 50  │ 60  │
                     └─────┴─────┴─────┴─────┴─────┴─────┴─────┘
                     ┌─────┬─────┬─────┬─────┬─────┬─────┐
3. pop():            │  5  │ 10  │ 20  │ 30  │ 40  │ 50  │
                     └─────┴─────┴─────┴─────┴─────┘
                           ┌─────┬─────┬─────┬─────┬─────┐
4. popleft():              │ 10  │ 20  │ 30  │ 40  │ 50  │
                           └─────┴─────┴─────┴─────┴─────┘
```

### Inserting & Extending Elements

> **Gotcha:** `extendleft(iterable)` appends elements to the left **one at a time**, so the order of the iterable is **reversed** in the resulting deque!

In [2]:
my_deque = deque([1, 2, 3, 4, 5])

# O(1) access/modification at the ends (but O(n) in the middle!)
print(f"{my_deque[0] = }, {my_deque[-1] = }")
my_deque[0] = 10
print("after my_deque[0] = 10:", my_deque)

my_deque.append(6)          # O(1)
my_deque.appendleft(0)      # O(1)
print("after append & appendleft:", my_deque)

my_deque.extend([7, 8])             # O(k)
my_deque.extendleft([100, 1001])    # O(k) — note: 1001 ends up first!
print("after extend & extendleft:", my_deque)

my_deque[0] = 1, my_deque[-1] = 5
after my_deque[0] = 10: deque([10, 2, 3, 4, 5])
after append & appendleft: deque([0, 10, 2, 3, 4, 5, 6])
after extend & extendleft: deque([1001, 100, 0, 10, 2, 3, 4, 5, 6, 7, 8])


### Removing, Counting, Indexing, Reversing & Rotating

- `pop()` / `popleft()` are $O(1)$ and return the removed element.
- `remove(val)` is $O(n)$, removes the **first** occurrence, and returns `None` (raises `ValueError` if missing).
- `reverse()` reverses the deque **in-place in $O(n)$ time** (it swaps elements from both ends inward).
- `rotate(k)` shifts elements right by `k` steps (or left if `k < 0`) in $O(\min(|k|, n - |k|))$.
- There is **no** `.sort()` method on `deque`; use `sorted(my_deque)` if needed.

In [3]:
d = deque([1, 2, 2, 3, 2, 4])

print("remove(3) returns:", d.remove(3), "-> deque:", d)
print("pop() returns:", d.pop(), "-> deque:", d)
print("popleft() returns:", d.popleft(), "-> deque:", d)

d2 = deque([1, 2, 2, 3, 2, 4])
print(f"{d2.count(2) = }")     # O(n)
print(f"{d2.index(4) = }")     # O(n) — first occurrence

d2.reverse()                   # O(n) in-place
print("reversed:", d2)

d2.rotate(2)                   # shift right by 2
print("rotate(+2):", d2)
d2.rotate(-2)                  # shift back left by 2
print("rotate(-2):", d2)

remove(3) returns: None -> deque: deque([1, 2, 2, 2, 4])
pop() returns: 4 -> deque: deque([1, 2, 2, 2])
popleft() returns: 1 -> deque: deque([2, 2, 2])
d2.count(2) = 3
d2.index(4) = 5
reversed: deque([4, 2, 3, 2, 2, 1])
rotate(+2): deque([2, 1, 4, 2, 3, 2])
rotate(-2): deque([4, 2, 3, 2, 2, 1])


### Bounded `deque(maxlen=k)` — Fixed-Size Ring Buffer

Setting `maxlen` automatically evicts items from the opposite end when full in $O(1)$ time. This is ideal for tracking the last $k$ log lines, moving averages, or sliding windows.

In [4]:
recent = deque(maxlen=3)
for x in [10, 20, 30, 40, 50]:
    recent.append(x)
    print(f"pushed {x} -> {list(recent)}")

pushed 10 -> [10]
pushed 20 -> [10, 20]
pushed 30 -> [10, 20, 30]
pushed 40 -> [20, 30, 40]
pushed 50 -> [30, 40, 50]


---
## 2. `Counter` (Subclass of `dict`)

`Counter` maps **hashable** elements to integer counts.

Key properties:
- Missing keys return **`0`** without raising `KeyError` and **without inserting the key** (unlike `defaultdict(int)`).
- Supports `most_common(k)` (uses `heapq.nlargest` internally -> $O(n \log k)$).
- Supports **multiset arithmetic**: `+` (add), `-` (subtract, drops $\le 0$), `&` (intersection / min), `|` (union / max), and `.total()` (Python 3.10+).
- Keys must be **hashable** (e.g., `int`, `str`, `tuple` of hashables — not `list` or `dict`).

In [5]:
assert issubclass(Counter, dict)

# Unhashable items (lists) raise TypeError
data = [["apple", "red"], ["banana", "yellow"], ["apple", "red"], ["grape", "purple"]]
try:
    Counter(data)
except TypeError as e:
    print(f"TypeError as expected: {e}")

# Fix: convert inner lists to hashable tuples
counts = Counter(tuple(item) for item in data)
print("Counts:", counts)
print("Most common (top 2):", counts.most_common(2))
print("Missing key returns 0 without mutating:", counts[("pear", "green")], "| still in counter?", ("pear", "green") in counts)

# Multiset arithmetic
a = Counter("abracadabra")
b = Counter("alacazam")
print("a + b:", a + b)
print("a - b (positive counts only):", a - b)
print("a & b (min / intersection):", a & b)
print("a | b (max / union):", a | b)
print("total elements in a:", a.total())

TypeError as expected: unhashable type: 'list'
Counts: Counter({('apple', 'red'): 2, ('banana', 'yellow'): 1, ('grape', 'purple'): 1})
Most common (top 2): [(('apple', 'red'), 2), (('banana', 'yellow'), 1)]
Missing key returns 0 without mutating: 0 | still in counter? False
a + b: Counter({'a': 9, 'b': 2, 'r': 2, 'c': 2, 'd': 1, 'l': 1, 'z': 1, 'm': 1})
a - b (positive counts only): Counter({'b': 2, 'r': 2, 'a': 1, 'd': 1})
a & b (min / intersection): Counter({'a': 4, 'c': 1})
a | b (max / union): Counter({'a': 5, 'b': 2, 'r': 2, 'c': 1, 'd': 1, 'l': 1, 'z': 1, 'm': 1})
total elements in a: 11


---
## 3. `OrderedDict` (Subclass of `dict`)

> **Important Python 3.7+ clarification:** Standard `dict` **is guaranteed by the Python language specification** to preserve insertion order (and overwriting an existing key keeps its original position).

So when do you still need `OrderedDict`?
1. **`move_to_end(key, last=True)`** — $O(1)$ repositioning to either end (the backbone of an **LRU Cache**).
2. **`popitem(last=True)`** — $O(1)$ pop from **either** end (`last=False` pops FIFO; regular `dict.popitem()` is LIFO-only).
3. **Order-sensitive equality** — two `OrderedDict`s with the same pairs in different orders compare `False`, whereas plain `dict`s compare `True`.

In [6]:
assert issubclass(OrderedDict, dict)

# 1. Order-sensitive equality
d1, d2 = {"a": 1, "b": 2}, {"b": 2, "a": 1}
od1, od2 = OrderedDict([("a", 1), ("b", 2)]), OrderedDict([("b", 2), ("a", 1)])
print(f"Plain dict equality: {d1 == d2}")
print(f"OrderedDict equality: {od1 == od2}")

# 2. move_to_end & popitem(last=False)
od = OrderedDict([("a", 1), ("b", 2), ("c", 3)])
od.move_to_end("a")                 # move 'a' to the right end: b, c, a
print("After move_to_end('a'):", list(od.keys()))
od.move_to_end("c", last=False)     # move 'c' to the front: c, b, a
print("After move_to_end('c', last=False):", list(od.keys()))
print("FIFO popitem(last=False):", od.popitem(last=False), "-> remaining:", list(od.keys()))

# 3. Classic O(1) LRU Cache in 15 lines
class LRUCache:
    def __init__(self, capacity: int):
        self.capacity = capacity
        self.cache: OrderedDict[str, int] = OrderedDict()

    def get(self, key: str) -> int:
        if key not in self.cache:
            return -1
        self.cache.move_to_end(key)
        return self.cache[key]

    def put(self, key: str, value: int) -> None:
        self.cache[key] = value
        self.cache.move_to_end(key)
        if len(self.cache) > self.capacity:
            self.cache.popitem(last=False)

lru = LRUCache(2)
lru.put("x", 1)
lru.put("y", 2)
_ = lru.get("x")    # 'x' becomes most-recently used
lru.put("z", 3)     # evicts 'y'
print("LRU state after inserting z:", list(lru.cache.items()))

Plain dict equality: True
OrderedDict equality: False
After move_to_end('a'): ['b', 'c', 'a']
After move_to_end('c', last=False): ['c', 'b', 'a']
FIFO popitem(last=False): ('c', 3) -> remaining: ['b', 'a']
LRU state after inserting z: [('x', 1), ('z', 3)]


---
## 4. `defaultdict` (Subclass of `dict`)

`defaultdict(default_factory)` calls `default_factory()` whenever a missing key is accessed via `d[key]`, avoiding boilerplate `if key not in d:` checks.

> **Gotcha:** Simply **reading** `d[missing_key]` mutates the dictionary by inserting `missing_key: default_factory()`. Use `d.get(key)` or `key in d` when checking membership without side effects!

In [7]:
assert issubclass(defaultdict, dict)

# Grouping items with defaultdict(list)
colors = [("yellow", 1), ("blue", 2), ("yellow", 3), ("blue", 4), ("red", 1)]
grouped = defaultdict(list)
for k, v in colors:
    grouped[k].append(v)

# Membership check WITHOUT mutating
print("'purple' in grouped:", "purple" in grouped, "| grouped.get('purple'):", grouped.get("purple"))

# Accessing via [] DOES insert a default value!
_ = grouped["green"]
print("After reading grouped['green']:", dict(grouped))

# Nested defaultdict for 2D counting (e.g., bigram counts)
bigrams = defaultdict(lambda: defaultdict(int))
for a, b in [("the", "cat"), ("the", "cat"), ("the", "dog")]:
    bigrams[a][b] += 1
print("Bigrams:", {k: dict(v) for k, v in bigrams.items()})

'purple' in grouped: False | grouped.get('purple'): None
After reading grouped['green']: {'yellow': [1, 3], 'blue': [2, 4], 'red': [1], 'green': []}
Bigrams: {'the': {'cat': 2, 'dog': 1}}


---
## 5. `namedtuple`

`namedtuple` generates lightweight, **immutable** `tuple` subclasses with named fields. They use far less memory than standard classes (no per-instance `__dict__`), remain hashable, support tuple unpacking, and provide `_replace()` and `_asdict()`.

In [8]:
Point = namedtuple("Point", ["x", "y"], defaults=[0])

p1 = Point(3, 4)
p2 = Point(x=5)                  # uses default y=0
p3 = p1._replace(y=10)           # returns a new Point (immutable)

x, y = p1                        # tuple unpacking still works
print(f"{p1 = }, unpacked: ({x}, {y})")
print(f"{p2 = }")
print(f"{p3 = }, asdict: {p3._asdict()}")

p1 = Point(x=3, y=4), unpacked: (3, 4)
p2 = Point(x=5, y=0)
p3 = Point(x=3, y=10), asdict: {'x': 3, 'y': 10}


---
## 6. `ChainMap`

`ChainMap(*maps)` groups multiple dictionaries into a **single logical view** without copying data. Lookups search maps from left to right; writes/deletes affect **only the first map**. Classic use case: layering CLI args -> environment variables -> defaults.

In [9]:
defaults = {"lr": 1e-3, "batch_size": 32, "device": "cpu"}
env_cfg = {"device": "cuda"}
cli_args = {"batch_size": 64}

config = ChainMap(cli_args, env_cfg, defaults)
print("Effective config:", dict(config))
print("batch_size (from cli):", config["batch_size"])
print("device (from env):", config["device"])
print("lr (from defaults):", config["lr"])

# Mutations only touch the first mapping (cli_args)
config["epochs"] = 10
print("cli_args after mutation:", cli_args)
print("defaults unchanged:", defaults)

Effective config: {'lr': 0.001, 'batch_size': 64, 'device': 'cuda'}
batch_size (from cli): 64
device (from env): cuda
lr (from defaults): 0.001
cli_args after mutation: {'batch_size': 64, 'epochs': 10}
defaults unchanged: {'lr': 0.001, 'batch_size': 32, 'device': 'cpu'}
